In [ ]:
# In[1]:

import pandas as pd
import numpy as np

# Load CSVs and parse timestamps as UTC datetimes
df_metric = pd.read_csv("metric.csv")
df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

df_trace = pd.read_csv("trace.csv")
df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

df_log = pd.read_csv("log.csv")
df_log['timestamp'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)

df_err = pd.read_csv("error_logs.csv")
df_err['timestamp'] = pd.to_datetime(df_err['timestamp'], unit='s', utc=True)

# Ensure value columns are numeric where present
for df in (df_metric, df_trace, df_log):
    if 'value' in df.columns:
        df['value'] = pd.to_numeric(df['value'], errors='coerce')

# Aggregate metric.csv by (cmdb_id, kpi_name)
metric_agg = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(
        count=('value', 'count'),
        min_timestamp=('timestamp', 'min'),
        max_timestamp=('timestamp', 'max'),
        p95_value=('value', lambda x: x.quantile(0.95) if x.notna().any() else np.nan)
    )
    .sort_values('count', ascending=False)
)

metric_agg_head = metric_agg[['cmdb_id', 'kpi_name', 'count', 'min_timestamp', 'max_timestamp', 'p95_value']].head(50)

# Aggregate trace.csv by (cmdb_id, trace_name)
trace_agg = (
    df_trace
    .groupby(['cmdb_id', 'trace_name'], as_index=False)
    .agg(
        count=('value', 'count'),
        min_timestamp=('timestamp', 'min'),
        max_timestamp=('timestamp', 'max'),
        p95_value=('value', lambda x: x.quantile(0.95) if x.notna().any() else np.nan)
    )
    .sort_values('count', ascending=False)
)

trace_agg_head = trace_agg[['cmdb_id', 'trace_name', 'count', 'min_timestamp', 'max_timestamp', 'p95_value']].head(50)

# Aggregate log.csv by (cmdb_id, log_name)
log_agg = (
    df_log
    .groupby(['cmdb_id', 'log_name'], as_index=False)
    .agg(
        count=('value', 'count'),
        min_timestamp=('timestamp', 'min'),
        max_timestamp=('timestamp', 'max'),
        p95_value=('value', lambda x: x.quantile(0.95) if x.notna().any() else np.nan)
    )
    .sort_values('count', ascending=False)
)

log_agg_head = log_agg[['cmdb_id', 'log_name', 'count', 'min_timestamp', 'max_timestamp', 'p95_value']].head(50)

# error_logs.csv: per-cmdb_id counts and earliest/latest timestamps
error_summary = (
    df_err
    .groupby('cmdb_id', as_index=False)
    .agg(
        count=('message', 'count'),
        first_timestamp=('timestamp', 'min'),
        last_timestamp=('timestamp', 'max')
    )
    .sort_values('count', ascending=False)
)

# up to 5 example error rows (timestamp, cmdb_id, message) - show the latest 5
error_examples = df_err.loc[:, ['timestamp', 'cmdb_id', 'message']].sort_values('timestamp', ascending=False).head(5)

# Display compact outputs (variables)
metric_agg_head, trace_agg_head, log_agg_head, error_summary, error_examples

```
Out[1]:
```
summary = (
    "Telemetry summary (UTC):\n"
    "- Metrics: adservice shows strong resource signals: very high disk I/O (p95 ~9.2e4) and large memory values (p95 ~1.07e8); CPU and socket counts are non‑zero as well.\n"
    "- Traces: checkoutservice trace series are present and active (multiple duration and row_count metrics), but trace latencies shown are relatively small.\n"
    "- Logs: frontend log.error_count is elevated (p95 64) and there are many recent error log entries.\n"
    "- Error logs: frontend generated 178 error messages in a short window (earliest 2024-02-21 00:31:56+00:00 to latest 2024-02-21 00:32:13+00:00) with repeated \"request error\" messages.\n\n"
    "Top-level conclusion: the most immediate failure signal is the frontend service (large batch of request errors). Adservice shows notable resource pressure (disk I/O and memory) and should be investigated as a potential contributing factor."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(           cmdb_id    kpi_name  count             min_timestamp             max_timestamp     p95_value
3        adservice      diskio     16 2024-02-21 00:17:00+00:00 2024-02-21 00:32:00+00:00  9.194913e+04
2        adservice         cpu     16 2024-02-21 00:17:00+00:00 2024-02-21 00:32:00+00:00  8.360459e-01
6        adservice         mem     16 2024-02-21 00:17:00+00:00 2024-02-21 00:32:00+00:00  1.071611e+08
7        adservice      socket     16 2024-02-21 00:17:00+00:00 2024-02-21 00:32:00+00:00  4.000000e+00
4        adservice  latency-50     16 2024-02-21 00:17:00+00:00 2024-02-21 00:32:00+00:00  3.003402e-03
..             ...         ...    ...                       ...                       ...           ...
45  paymentservice  latency-50     16 2024-02-21 00:17:00+00:00 2024-02-21 00:32:00+00:00  5.895334e-03
46  paymentservice  latency-90     16 2024-02-21 00:17:00+00:00 2024-02-21 00:32:00+00:00  9.606621e-03
47  paymentservice         mem     16 2024-02-21 00:17:00+00:00 2024-02-21 00:32:00+00:00  3.464457e+07
48  paymentservice      socket     16 2024-02-21 00:17:00+00:00 2024-02-21 00:32:00+00:00  3.000000e+00
49  paymentservice    workload     16 2024-02-21 00:17:00+00:00 2024-02-21 00:32:00+00:00  1.276575e+00

[50 rows x 6 columns],             cmdb_id                                trace_name  count             min_timestamp             max_timestamp   p95_value
0   checkoutservice  trace.from_checkoutservice.duration_mean     13 2024-02-21 00:17:00+00:00 2024-02-21 00:29:00+00:00    0.007152
1   checkoutservice   trace.from_checkoutservice.duration_p95     13 2024-02-21 00:17:00+00:00 2024-02-21 00:29:00+00:00    0.014327
2   checkoutservice     trace.from_checkoutservice.error_rate     13 2024-02-21 00:17:00+00:00 2024-02-21 00:29:00+00:00    0.000000
3   checkoutservice      trace.from_checkoutservice.row_count     13 2024-02-21 00:17:00+00:00 2024-02-21 00:29:00+00:00  337.600000
4   checkoutservice  trace.from_frontendservice.duration_mean     13 2024-02-21 00:17:00+00:00 2024-02-21 00:29:00+00:00    0.125085
..              ...                                       ...    ...                       ...                       ...         ...
45     emailservice   trace.from_checkoutservice.duration_p95     13 2024-02-21 00:17:00+00:00 2024-02-21 00:29:00+00:00    0.000828
46     emailservice     trace.from_checkoutservice.error_rate     13 2024-02-21 00:17:00+00:00 2024-02-21 00:29:00+00:00    0.000000
47     emailservice      trace.from_checkoutservice.row_count     13 2024-02-21 00:17:00+00:00 2024-02-21 00:29:00+00:00   41.600000
63  frontendservice        trace.to_checkoutservice.row_count     13 2024-02-21 00:17:00+00:00 2024-02-21 00:29:00+00:00   41.600000
62  frontendservice       trace.to_checkoutservice.error_rate     13 2024-02-21 00:17:00+00:00 2024-02-21 00:29:00+00:00    0.000000

[50 rows x 6 columns],                   cmdb_id         log_name  count             min_timestamp             max_timestamp  p95_value
0               adservice  log.error_count     16 2024-02-21 00:17:00+00:00 2024-02-21 00:32:00+00:00       0.00
1               adservice    log.row_count     16 2024-02-21 00:17:00+00:00 2024-02-21 00:32:00+00:00     465.25
2             cartservice  log.error_count     16 2024-02-21 00:17:00+00:00 2024-02-21 00:32:00+00:00       0.00
3             cartservice    log.row_count     16 2024-02-21 00:17:00+00:00 2024-02-21 00:32:00+00:00     824.75
10               frontend  log.error_count     16 2024-02-21 00:17:00+00:00 2024-02-21 00:32:00+00:00      64.00
11               frontend    log.row_count     16 2024-02-21 00:17:00+00:00 2024-02-21 00:32:00+00:00    2792.25
4         checkoutservice  log.error_count     15 2024-02-21 00:17:00+00:00 2024-02-21 00:31:00+00:00       0.00
5         checkoutservice    log.row_count     15 2024-02-21 00:17:00+00:00 2024-02-21 00:31:00+00:00     123.60
7         currencyservice    log.row_count     15 2024-02-21 00:17:00+00:00 2024-02-21 00:31:00+00:00    2457.80
6         currencyservice  log.error_count     15 2024-02-21 00:17:00+00:00 2024-02-21 00:31:00+00:00       0.00
9            emailservice    log.row_count     15 2024-02-21 00:17:00+00:00 2024-02-21 00:31:00+00:00      41.20
8            emailservice  log.error_count     15 2024-02-21 00:17:00+00:00 2024-02-21 00:31:00+00:00       0.00
12         paymentservice  log.error_count     15 2024-02-21 00:17:00+00:00 2024-02-21 00:31:00+00:00       0.00
13         paymentservice    log.row_count     15 2024-02-21 00:17:00+00:00 2024-02-21 00:31:00+00:00      82.40
14  recommendationservice  log.error_count     15 2024-02-21 00:17:00+00:00 2024-02-21 00:31:00+00:00       0.00
15  recommendationservice    log.row_count     15 2024-02-21 00:17:00+00:00 2024-02-21 00:31:00+00:00     605.20
19        shippingservice    log.row_count     15 2024-02-21 00:17:00+00:00 2024-02-21 00:31:00+00:00     530.20
18        shippingservice  log.error_count     15 2024-02-21 00:17:00+00:00 2024-02-21 00:31:00+00:00       0.00
17                  redis    log.row_count      4 2024-02-21 00:21:00+00:00 2024-02-21 00:32:00+00:00       5.00
16                  redis  log.error_count      4 2024-02-21 00:21:00+00:00 2024-02-21 00:32:00+00:00       0.00,     cmdb_id  count           first_timestamp            last_timestamp
0  frontend    178 2024-02-21 00:31:56+00:00 2024-02-21 00:32:13+00:00,                     timestamp   cmdb_id        message
177 2024-02-21 00:32:13+00:00  frontend  request error
176 2024-02-21 00:32:12+00:00  frontend  request error
175 2024-02-21 00:32:12+00:00  frontend  request error
174 2024-02-21 00:32:12+00:00  frontend  request error
173 2024-02-21 00:32:12+00:00  frontend  request error)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# Reuse existing df_metric if available; otherwise load and parse timestamps
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")
    df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# Ensure timestamp parsed and value numeric
df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

# 1-2) Compute global P95 per (cmdb_id, kpi_name) using the full series (global threshold)
global_p95 = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'global_p95'})
)

# Merge global_p95 back to full metric series
df_metric = df_metric.merge(global_p95, on=['cmdb_id', 'kpi_name'], how='left')

# 3) Filter metric.csv to the incident window (UTC)
start = pd.to_datetime("2024-02-21 00:14:21", utc=True)
end   = pd.to_datetime("2024-02-21 00:44:21", utc=True)
df_window = df_metric[(df_metric['timestamp'] >= start) & (df_metric['timestamp'] <= end)].copy()

# Total number of metric rows in the incident window
total_rows_in_window = df_window.shape[0]

# 4) For each (cmdb_id, kpi_name) within the window, count anomaly points where value >= global_p95
# Treat KPIs as "higher-is-worse" by applying value >= global_p95. (global_p95 computed globally above)
df_window['is_anomaly'] = False
mask_valid = df_window['value'].notna() & df_window['global_p95'].notna()
df_window.loc[mask_valid, 'is_anomaly'] = df_window.loc[mask_valid, 'value'] >= df_window.loc[mask_valid, 'global_p95']

# Aggregate required fields per group within window
agg = (
    df_window
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(
        anomaly_count=('is_anomaly', 'sum'),
        earliest_anomaly_timestamp=('timestamp', lambda x: x[df_window.loc[x.index, 'is_anomaly']].min() if x.notna().any() else pd.NaT),
        max_value_in_window=('value', 'max'),
        global_p95=('global_p95', 'first')  # same for group
    )
)

# Ensure earliest_anomaly_timestamp is NaT where anomaly_count == 0
agg.loc[agg['anomaly_count'] == 0, 'earliest_anomaly_timestamp'] = pd.NaT

# 5) Return compact aggregated table sorted and limited to top 20 rows
agg_sorted = agg.sort_values(by=['anomaly_count', 'max_value_in_window'], ascending=[False, False])
metric_anomalies_head = agg_sorted[['cmdb_id', 'kpi_name', 'anomaly_count', 'earliest_anomaly_timestamp', 'max_value_in_window', 'global_p95']].head(20)

# Display compact results: top anomalies and total row count in window
metric_anomalies_head, total_rows_in_window

```
Out[2]:
```
summary = (
    "Compact summary (UTC):\n"
    "- Total metric rows in incident window: 1,154.\n"
    "- Dominant anomaly signal: socket KPIs across many services. Frontend.socket has the highest impact (16 anomaly points, earliest 2024-02-21 00:17:00, max=26, global_p95=26). Multiple services (adservice, redis, shippingservice, cartservice, currencyservice, productcatalogservice, recommendationservice, paymentservice, emailservice, etc.) show 15–16 socket anomalies.\n"
    "- Disk I/O groups show anomaly counts but their max values are 0 (global_p95=0) — likely not a meaningful spike.\n"
    "- Memory: a few single-point memory anomalies (e.g., adservice at 2024-02-21 00:28:00 slightly above global P95), but these are isolated (anomaly_count=1).\n\n"
    "Conclusion: the most significant and widespread metric issue during the window is socket-related (possible socket exhaustion or high concurrent connections), with the frontend.socket being the clearest hotspot. Follow-up: investigate socket usage/limits and network connections on frontend and the identified services."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id kpi_name  anomaly_count earliest_anomaly_timestamp  max_value_in_window    global_p95
40               frontend   socket             16  2024-02-21 00:17:00+00:00         2.600000e+01  2.600000e+01
7               adservice   socket             16  2024-02-21 00:17:00+00:00         4.000000e+00  4.000000e+00
67                  redis   socket             16  2024-02-21 00:17:00+00:00         4.000000e+00  4.000000e+00
72        shippingservice   socket             16  2024-02-21 00:17:00+00:00         3.000000e+00  3.000000e+00
16        checkoutservice   diskio             16  2024-02-21 00:17:00+00:00         0.000000e+00  0.000000e+00
23        currencyservice    error             16  2024-02-21 00:17:00+00:00         0.000000e+00  0.000000e+00
30           emailservice   diskio             16  2024-02-21 00:17:00+00:00         0.000000e+00  0.000000e+00
51  productcatalogservice   diskio             16  2024-02-21 00:17:00+00:00         0.000000e+00  0.000000e+00
58  recommendationservice   diskio             16  2024-02-21 00:17:00+00:00         0.000000e+00  0.000000e+00
13            cartservice   socket             15  2024-02-21 00:17:00+00:00         4.000000e+00  4.000000e+00
27        currencyservice   socket             15  2024-02-21 00:17:00+00:00         4.000000e+00  4.000000e+00
55  productcatalogservice   socket             15  2024-02-21 00:17:00+00:00         4.000000e+00  4.000000e+00
62  recommendationservice   socket             15  2024-02-21 00:17:00+00:00         4.000000e+00  4.000000e+00
34           emailservice   socket             15  2024-02-21 00:17:00+00:00         3.000000e+00  3.000000e+00
48         paymentservice   socket             15  2024-02-21 00:17:00+00:00         3.000000e+00  3.000000e+00
6               adservice      mem              1  2024-02-21 00:28:00+00:00         1.073674e+08  1.071611e+08
26        currencyservice      mem              1  2024-02-21 00:30:00+00:00         4.554772e+07  4.535624e+07
61  recommendationservice      mem              1  2024-02-21 00:17:00+00:00         4.529115e+07  4.527991e+07
33           emailservice      mem              1  2024-02-21 00:32:00+00:00         4.362200e+07  4.361923e+07
12            cartservice      mem              1  2024-02-21 00:18:00+00:00         4.332346e+07  4.330962e+07, 1154)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reuse df_trace if present, otherwise load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")

# Parse timestamp and ensure value numeric
df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')

# Define incident window (UTC)
start = pd.to_datetime("2024-02-21 00:14:21", utc=True)
end   = pd.to_datetime("2024-02-21 00:44:21", utc=True)

# Filter to window
df_trace_window = df_trace[(df_trace['timestamp'] >= start) & (df_trace['timestamp'] <= end)].copy()

# 1) Aggregate by (cmdb_id, trace_name) within the window
trace_agg = (
    df_trace_window
    .groupby(['cmdb_id', 'trace_name'], as_index=False)
    .agg(
        count=('value', 'count'),
        earliest_timestamp=('timestamp', 'min'),
        latest_timestamp=('timestamp', 'max'),
        max_value_in_window=('value', 'max'),
        p95_value_in_window=('value', lambda x: x.dropna().quantile(0.95) if x.dropna().size>0 else np.nan)
    )
    .sort_values(by='max_value_in_window', ascending=False)
)

trace_agg_top20 = trace_agg[['cmdb_id', 'trace_name', 'count', 'earliest_timestamp', 'latest_timestamp', 'max_value_in_window', 'p95_value_in_window']].head(20)

# 2) Rows where trace_name contains 'error_rate' AND value > 0, sorted by timestamp asc (limit 50)
mask_error_rate = df_trace_window['trace_name'].str.contains('error_rate', case=False, na=False) & (df_trace_window['value'] > 0)
trace_error_rows = df_trace_window.loc[mask_error_rate, ['timestamp', 'cmdb_id', 'trace_name', 'value']].sort_values('timestamp', ascending=True).head(50)

# 3) Rows where trace_name contains 'duration' AND (cmdb_id contains 'frontend' OR trace_name contains 'to_checkoutservice' OR 'from_checkoutservice')
mask_duration = (
    df_trace_window['trace_name'].str.contains('duration', case=False, na=False) &
    (
        df_trace_window['cmdb_id'].str.contains('frontend', case=False, na=False) |
        df_trace_window['trace_name'].str.contains('to_checkoutservice', case=False, na=False) |
        df_trace_window['trace_name'].str.contains('from_checkoutservice', case=False, na=False)
    )
)
trace_duration_rows = df_trace_window.loc[mask_duration, ['timestamp', 'cmdb_id', 'trace_name', 'value']].sort_values('value', ascending=False).head(50)

# Return compact results
trace_agg_top20, trace_error_rows, trace_duration_rows

```
Out[3]:
```
summary = (
    "Trace analysis (UTC 2024-02-21 00:14:21 → 00:44:21):\n"
    "- Top aggregated traces (by max value): many high row_count traces, led by frontendservice (to_frontendservice & from_frontendservice) with max row_count ~8190 and p95 ~8155 — frontend is the dominant traffic hub.\n"
    "- Error rates: no trace rows with trace_name containing 'error_rate' had value > 0 in the window (no non‑zero trace error_rate found).\n"
    "- Durations of concern: frontend-related duration metrics (especially trace.to_checkoutservice.duration_p95) show the highest latency values in-window — peak p95 ≈ 0.305s, with other frontend duration p95 values around 0.27–0.20s.\n\n"
    "Conclusion: frontend is the primary source of traffic and shows the largest row_count volumes. While trace error rates are not elevated, frontend→checkoutservice duration p95 is the largest latency signal (≈0.3s) and should be investigated along with frontend load patterns."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                   cmdb_id                                  trace_name  count        earliest_timestamp          latest_timestamp  max_value_in_window  p95_value_in_window
71         frontendservice          trace.to_frontendservice.row_count     13 2024-02-21 00:17:00+00:00 2024-02-21 00:29:00+00:00               8190.0               8155.2
55         frontendservice        trace.from_frontendservice.row_count     13 2024-02-21 00:17:00+00:00 2024-02-21 00:29:00+00:00               8190.0               8155.2
95   productcatalogservice        trace.from_frontendservice.row_count     12 2024-02-21 00:17:00+00:00 2024-02-21 00:29:00+00:00               3907.0               3894.9
75         frontendservice    trace.to_productcatalogservice.row_count     12 2024-02-21 00:17:00+00:00 2024-02-21 00:29:00+00:00               3907.0               3894.9
39         currencyservice        trace.from_frontendservice.row_count     13 2024-02-21 00:17:00+00:00 2024-02-21 00:29:00+00:00               2421.0               2351.4
67         frontendservice          trace.to_currencyservice.row_count     13 2024-02-21 00:17:00+00:00 2024-02-21 00:29:00+00:00               2421.0               2351.4
59         frontendservice                   trace.from_root.row_count     13 2024-02-21 00:17:00+00:00 2024-02-21 00:29:00+00:00                863.0                854.0
139                   root          trace.to_frontendservice.row_count     13 2024-02-21 00:17:00+00:00 2024-02-21 00:29:00+00:00                863.0                854.0
119  recommendationservice    trace.to_productcatalogservice.row_count     12 2024-02-21 00:17:00+00:00 2024-02-21 00:29:00+00:00                608.0                605.8
107  recommendationservice        trace.from_frontendservice.row_count     13 2024-02-21 00:17:00+00:00 2024-02-21 00:29:00+00:00                608.0                605.6
111  recommendationservice  trace.from_recommendationservice.row_count     13 2024-02-21 00:17:00+00:00 2024-02-21 00:29:00+00:00                608.0                605.6
79         frontendservice    trace.to_recommendationservice.row_count     13 2024-02-21 00:17:00+00:00 2024-02-21 00:29:00+00:00                608.0                605.6
123  recommendationservice    trace.to_recommendationservice.row_count     13 2024-02-21 00:17:00+00:00 2024-02-21 00:29:00+00:00                608.0                605.6
99   productcatalogservice  trace.from_recommendationservice.row_count     12 2024-02-21 00:17:00+00:00 2024-02-21 00:29:00+00:00                608.0                605.8
3          checkoutservice        trace.from_checkoutservice.row_count     13 2024-02-21 00:17:00+00:00 2024-02-21 00:29:00+00:00                340.0                337.6
15         checkoutservice          trace.to_checkoutservice.row_count     13 2024-02-21 00:17:00+00:00 2024-02-21 00:29:00+00:00                340.0                337.6
11         checkoutservice                   trace.from_root.row_count     13 2024-02-21 00:17:00+00:00 2024-02-21 00:29:00+00:00                160.0                150.4
127                   root          trace.to_checkoutservice.row_count     13 2024-02-21 00:17:00+00:00 2024-02-21 00:29:00+00:00                160.0                150.4
35         currencyservice        trace.from_checkoutservice.row_count     13 2024-02-21 00:17:00+00:00 2024-02-21 00:29:00+00:00                136.0                126.4
19         checkoutservice          trace.to_currencyservice.row_count     13 2024-02-21 00:17:00+00:00 2024-02-21 00:29:00+00:00                136.0                126.4, Empty DataFrame
Columns: [timestamp, cmdb_id, trace_name, value]
Index: [],                      timestamp          cmdb_id                             trace_name     value
1829 2024-02-21 00:29:00+00:00  frontendservice  trace.to_checkoutservice.duration_p95  0.305248
1577 2024-02-21 00:27:00+00:00  frontendservice           trace.from_root.duration_p95  0.270226
669  2024-02-21 00:21:00+00:00  frontendservice  trace.to_checkoutservice.duration_p95  0.207567
213  2024-02-21 00:18:00+00:00  frontendservice  trace.to_checkoutservice.duration_p95  0.198251
1277 2024-02-21 00:25:00+00:00  frontendservice  trace.to_checkoutservice.duration_p95  0.187158
...                        ...              ...                                    ...       ...
1645 2024-02-21 00:27:00+00:00             root  trace.to_checkoutservice.duration_p95  0.067225
581  2024-02-21 00:20:00+00:00             root  trace.to_checkoutservice.duration_p95  0.066918
664  2024-02-21 00:21:00+00:00  frontendservice          trace.from_root.duration_mean  0.066840
1037 2024-02-21 00:23:00+00:00             root  trace.to_checkoutservice.duration_p95  0.066412
360  2024-02-21 00:19:00+00:00  frontendservice          trace.from_root.duration_mean  0.064960

[50 rows x 4 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Load or reuse metric and trace dataframes, ensure timestamps parsed and values numeric
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")
df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")
df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')

# Incident window (UTC)
start = pd.to_datetime("2024-02-21 00:14:21", utc=True)
end   = pd.to_datetime("2024-02-21 00:44:21", utc=True)

# Candidate services
candidates = ['checkoutservice', 'currencyservice', 'emailservice', 'productcatalogservice', 'recommendationservice']

# 1) Compute global P95 per (cmdb_id, kpi_name) using the full metric series (global, before window filtering)
global_p95_df = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(global_p95=('value', lambda x: x.dropna().quantile(0.95) if x.dropna().size>0 else np.nan))
)

# Merge global_p95 into df_metric (ensures column exists)
df_metric = df_metric.merge(global_p95_df, on=['cmdb_id', 'kpi_name'], how='left')

# Filter metric to the incident window
df_metric_window = df_metric[(df_metric['timestamp'] >= start) & (df_metric['timestamp'] <= end)].copy()

# Filter to candidate services
df_metric_cand = df_metric_window[df_metric_window['cmdb_id'].isin(candidates)].copy()

# Helper: compute max consecutive True length in time-ordered boolean series
def max_consecutive_true_bool(s: pd.Series) -> int:
    if s.empty:
        return 0
    arr = s.to_numpy(dtype=np.bool_)
    if not arr.any():
        return 0
    max_len = 0
    cur = 0
    for v in arr:
        if v:
            cur += 1
            if cur > max_len:
                max_len = cur
        else:
            cur = 0
    return int(max_len)

# Determine anomaly boolean: higher-is-worse -> value >= global_p95
df_metric_cand['is_anomaly'] = False
mask_valid = df_metric_cand['value'].notna() & df_metric_cand['global_p95'].notna()
df_metric_cand.loc[mask_valid, 'is_anomaly'] = df_metric_cand.loc[mask_valid, 'value'] >= df_metric_cand.loc[mask_valid, 'global_p95']

# Aggregate per (cmdb_id, kpi_name) for candidate services
agg_rows = []
for (cmdb_id, kpi_name), grp in df_metric_cand.groupby(['cmdb_id', 'kpi_name'], sort=False):
    grp_sorted = grp.sort_values('timestamp')
    anomaly_count = int(grp_sorted['is_anomaly'].sum())
    earliest_anom_ts = grp_sorted.loc[grp_sorted['is_anomaly'], 'timestamp'].min() if anomaly_count > 0 else pd.NaT
    max_value_in_window = grp_sorted['value'].max()
    global_p95_val = grp_sorted['global_p95'].iloc[0] if 'global_p95' in grp_sorted.columns else np.nan
    max_consec = max_consecutive_true_bool(grp_sorted['is_anomaly'])
    agg_rows.append({
        'cmdb_id': cmdb_id,
        'kpi_name': kpi_name,
        'anomaly_count': anomaly_count,
        'earliest_anomaly_timestamp': earliest_anom_ts,
        'max_value_in_window': max_value_in_window,
        'global_p95': global_p95_val,
        'max_consecutive_anomaly_length': max_consec
    })

metric_candidate_agg = pd.DataFrame(agg_rows)

# Sort and limit to top 50 rows by anomaly_count desc then cmdb_id
metric_candidate_agg_sorted = metric_candidate_agg.sort_values(by=['anomaly_count', 'cmdb_id'], ascending=[False, True])
metric_candidate_agg_head = metric_candidate_agg_sorted[['cmdb_id','kpi_name','anomaly_count','earliest_anomaly_timestamp','max_value_in_window','global_p95','max_consecutive_anomaly_length']].head(50)

# ---------------- TRACE ANALYSIS for candidate services ----------------

# Filter trace to window and to candidate services
df_trace_window = df_trace[(df_trace['timestamp'] >= start) & (df_trace['timestamp'] <= end)].copy()
df_trace_cand = df_trace_window[df_trace_window['cmdb_id'].isin(candidates)].copy()

# A) Aggregated table grouped by (cmdb_id, trace_name) for candidate services
trace_agg = (
    df_trace_cand
    .groupby(['cmdb_id', 'trace_name'], as_index=False)
    .agg(
        count=('value', 'count'),
        earliest_timestamp=('timestamp', 'min'),
        latest_timestamp=('timestamp', 'max'),
        max_value_in_window=('value', 'max'),
        p95_value_in_window=('value', lambda x: x.dropna().quantile(0.95) if x.dropna().size>0 else np.nan)
    )
    .sort_values(by='max_value_in_window', ascending=False)
)
trace_agg_top50 = trace_agg[['cmdb_id','trace_name','count','earliest_timestamp','latest_timestamp','max_value_in_window','p95_value_in_window']].head(50)

# B) trace rows where trace_name contains 'error_rate' AND value > 0 (limit 50), sorted by timestamp asc
mask_error_rate = df_trace_cand['trace_name'].str.contains('error_rate', case=False, na=False) & (df_trace_cand['value'] > 0)
trace_error_rows = df_trace_cand.loc[mask_error_rate, ['timestamp','cmdb_id','trace_name','value']].sort_values('timestamp', ascending=True).head(50)

# C) trace rows where trace_name contains 'duration' for candidate services, sorted by value desc (limit 50)
mask_duration = df_trace_cand['trace_name'].str.contains('duration', case=False, na=False)
trace_duration_rows = df_trace_cand.loc[mask_duration, ['timestamp','cmdb_id','trace_name','value']].sort_values('value', ascending=False).head(50)

# Return compact results
metric_candidate_agg_head, trace_agg_top50, trace_error_rows, trace_duration_rows

```
Out[4]:
```
summary = (
    "Summary (UTC, incident window 2024-02-21 00:14:21 → 00:44:21):\n\n"
    "Metrics (five candidate services: checkoutservice, currencyservice, emailservice, productcatalogservice, recommendationservice):\n"
    "- Many metric groups show anomaly_count=16 for diskio (and some other KPIs) but their max values and global P95 are 0 — these are not meaningful spikes but reflect a zero-valued KPI baseline.\n"
    "- A few resource anomalies are isolated single points (mem/cpu) for some services (anomaly_count=1), e.g., ad-hoc memory points slightly above global P95 — these are sparse, not sustained.\n"
    "- Overall, there is no sustained high-resource signal across these five services (most meaningful multi-point anomalies are zero-valued diskio entries).\n\n"
    "Trace telemetry (same window, five services):\n"
    "- The dominant signals are high row_count (traffic) traces: productcatalogservice and currencyservice receive heavy traffic from frontend (max row_count ~3907 and ~2421 respectively), recommendationservice and checkoutservice also show notable row_count (~608 and ~340).\n"
    "- No non-zero trace 'error_rate' rows were found for these five services in the window (no trace-level error-rate spikes).\n"
    "- Latency/duration: checkoutservice shows the largest duration p95 values (e.g., trace.from_frontendservice.duration_p95 peaked ~0.305s). Many checkout-related duration p95/mean entries are the top latency signals for these services.\n\n"
    "Conclusion and recommended next steps:\n"
    "- The five candidate services are experiencing increased request load (high row_count) coming from the frontend, especially to productcatalogservice and currencyservice; this is the principal signal.\n"
    "- Checkoutservice shows the clearest latency signal (p95 duration up to ~0.3s) and should be investigated for request handling/latency under load.\n"
    "- There is no trace-level error-rate elevation for these services, and most metric 'diskio' anomalies are zero-values and not actionable. Focus investigation on frontend→service traffic patterns and checkoutservice latency (investigate connection concurrency, request queueing, and service-level latency causes)."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id    kpi_name  anomaly_count earliest_anomaly_timestamp  max_value_in_window    global_p95  max_consecutive_anomaly_length
1         checkoutservice      diskio             16  2024-02-21 00:17:00+00:00         0.000000e+00  0.000000e+00                              16
8         currencyservice       error             16  2024-02-21 00:17:00+00:00         0.000000e+00  0.000000e+00                              16
15           emailservice      diskio             16  2024-02-21 00:17:00+00:00         0.000000e+00  0.000000e+00                              16
22  productcatalogservice      diskio             16  2024-02-21 00:17:00+00:00         0.000000e+00  0.000000e+00                              16
29  recommendationservice      diskio             16  2024-02-21 00:17:00+00:00         0.000000e+00  0.000000e+00                              16
..                    ...         ...            ...                        ...                  ...           ...                             ...
28  recommendationservice         cpu              1  2024-02-21 00:17:00+00:00         2.738143e+00  2.660673e+00                               1
30  recommendationservice  latency-50              1  2024-02-21 00:32:00+00:00         7.184040e-03  7.166056e-03                               1
31  recommendationservice  latency-90              1  2024-02-21 00:31:00+00:00         9.618260e-03  9.590348e-03                               1
32  recommendationservice         mem              1  2024-02-21 00:17:00+00:00         4.529115e+07  4.527991e+07                               1
34  recommendationservice    workload              1  2024-02-21 00:17:00+00:00         2.092813e+01  2.039457e+01                               1

[35 rows x 7 columns],                   cmdb_id                                  trace_name  count        earliest_timestamp          latest_timestamp  max_value_in_window  p95_value_in_window
59  productcatalogservice        trace.from_frontendservice.row_count     12 2024-02-21 00:17:00+00:00 2024-02-21 00:29:00+00:00          3907.000000          3894.900000
39        currencyservice        trace.from_frontendservice.row_count     13 2024-02-21 00:17:00+00:00 2024-02-21 00:29:00+00:00          2421.000000          2351.400000
71  recommendationservice        trace.from_frontendservice.row_count     13 2024-02-21 00:17:00+00:00 2024-02-21 00:29:00+00:00           608.000000           605.600000
63  productcatalogservice  trace.from_recommendationservice.row_count     12 2024-02-21 00:17:00+00:00 2024-02-21 00:29:00+00:00           608.000000           605.800000
83  recommendationservice    trace.to_productcatalogservice.row_count     12 2024-02-21 00:17:00+00:00 2024-02-21 00:29:00+00:00           608.000000           605.800000
..                    ...                                         ...    ...                       ...                       ...                  ...                  ...
24        checkoutservice       trace.to_paymentservice.duration_mean     13 2024-02-21 00:17:00+00:00 2024-02-21 00:29:00+00:00             0.000623             0.000546
44           emailservice    trace.from_checkoutservice.duration_mean     13 2024-02-21 00:17:00+00:00 2024-02-21 00:29:00+00:00             0.000511             0.000403
20        checkoutservice         trace.to_emailservice.duration_mean     13 2024-02-21 00:17:00+00:00 2024-02-21 00:29:00+00:00             0.000511             0.000403
36        currencyservice    trace.from_frontendservice.duration_mean     13 2024-02-21 00:17:00+00:00 2024-02-21 00:29:00+00:00             0.000446             0.000416
33        currencyservice     trace.from_checkoutservice.duration_p95     13 2024-02-21 00:17:00+00:00 2024-02-21 00:29:00+00:00             0.000384             0.000285

[50 rows x 7 columns], Empty DataFrame
Columns: [timestamp, cmdb_id, trace_name, value]
Index: [],                      timestamp                cmdb_id                               trace_name     value
1773 2024-02-21 00:29:00+00:00        checkoutservice  trace.from_frontendservice.duration_p95  0.305248
613  2024-02-21 00:21:00+00:00        checkoutservice  trace.from_frontendservice.duration_p95  0.207567
157  2024-02-21 00:18:00+00:00        checkoutservice  trace.from_frontendservice.duration_p95  0.198251
1221 2024-02-21 00:25:00+00:00        checkoutservice  trace.from_frontendservice.duration_p95  0.187158
461  2024-02-21 00:20:00+00:00        checkoutservice  trace.from_frontendservice.duration_p95  0.184247
...                        ...                    ...                                      ...       ...
768  2024-02-21 00:22:00+00:00        checkoutservice            trace.from_root.duration_mean  0.014030
1376 2024-02-21 00:26:00+00:00        checkoutservice            trace.from_root.duration_mean  0.013630
464  2024-02-21 00:20:00+00:00        checkoutservice            trace.from_root.duration_mean  0.013520
1528 2024-02-21 00:27:00+00:00        checkoutservice            trace.from_root.duration_mean  0.013439
405  2024-02-21 00:19:00+00:00  productcatalogservice             trace.from_root.duration_p95  0.013227

[50 rows x 4 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np

# Reuse existing dataframes if present, otherwise load from CSV
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")
try:
    df_log
except NameError:
    df_log = pd.read_csv("log.csv")
try:
    df_err
except NameError:
    df_err = pd.read_csv("error_logs.csv")

# Parse timestamps as UTC datetimes (Unix seconds)
df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)
df_log['timestamp'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)
df_err['timestamp'] = pd.to_datetime(df_err['timestamp'], unit='s', utc=True)

# Ensure numeric 'value' where applicable
if 'value' in df_metric.columns:
    df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
if 'value' in df_log.columns:
    df_log['value'] = pd.to_numeric(df_log['value'], errors='coerce')

# Incident window (UTC)
start = pd.to_datetime("2024-02-21 00:14:21", utc=True)
end   = pd.to_datetime("2024-02-21 00:44:21", utc=True)

# Filter to window
df_metric_window = df_metric[(df_metric['timestamp'] >= start) & (df_metric['timestamp'] <= end)].copy()
df_log_window = df_log[(df_log['timestamp'] >= start) & (df_log['timestamp'] <= end)].copy()
df_err_window = df_err[(df_err['timestamp'] >= start) & (df_err['timestamp'] <= end)].copy()

# 1) Unique metric kpi_name values for cmdb_id == 'frontend'
frontend_kpis = pd.DataFrame({
    'kpi_name': pd.Series(sorted(df_metric_window.loc[df_metric_window['cmdb_id']=='frontend', 'kpi_name'].dropna().unique()))
})

# 2) All metric rows for cmdb_id == 'frontend' and kpi_name == 'socket' within the window
frontend_socket_rows = (
    df_metric_window.loc[
        (df_metric_window['cmdb_id'] == 'frontend') & (df_metric_window['kpi_name'] == 'socket'),
        ['timestamp', 'value']
    ]
    .sort_values('timestamp', ascending=True)
    .head(100)
    .reset_index(drop=True)
)

# 3) Aggregated summary for frontend from log.csv within window
frontend_logs = df_log_window[df_log_window['cmdb_id'] == 'frontend'].copy()
if not frontend_logs.empty:
    frontend_log_agg = (
        frontend_logs
        .groupby('log_name', as_index=False)
        .agg(
            count=('value', 'count'),
            min_timestamp=('timestamp', 'min'),
            max_timestamp=('timestamp', 'max'),
            p95_value=('value', lambda x: x.dropna().quantile(0.95) if x.dropna().size>0 else np.nan)
        )
        .sort_values('count', ascending=False)
        .reset_index(drop=True)
    )
else:
    frontend_log_agg = pd.DataFrame(columns=['log_name','count','min_timestamp','max_timestamp','p95_value'])

# 4) Example error_logs for frontend: total count, earliest/latest timestamps, up to 20 example rows
frontend_errs = df_err_window[df_err_window['cmdb_id'] == 'frontend'].copy().sort_values('timestamp', ascending=True)
frontend_error_count = int(frontend_errs.shape[0])
frontend_error_first = frontend_errs['timestamp'].min() if frontend_error_count > 0 else pd.NaT
frontend_error_last  = frontend_errs['timestamp'].max() if frontend_error_count > 0 else pd.NaT
frontend_error_examples = frontend_errs.loc[:, ['timestamp', 'message']].head(20).reset_index(drop=True)

# Return compact outputs
frontend_kpis, frontend_socket_rows, frontend_log_agg, (frontend_error_count, frontend_error_first, frontend_error_last), frontend_error_examples

```
Out[5]:
```
Summary (UTC):

- Frontend metric KPIs present in-window: cpu, latency-50, latency-90, mem, socket, workload.

- Frontend socket metric (window 2024-02-21 00:17 → 00:32): 16 samples, each value = 26 (constant high socket count).

- Frontend logs (window):
  - log.error_count: 16 samples (min 00:17, max 00:32), p95 = 64.
  - log.row_count: 16 samples (min 00:17, max 00:32), p95 ≈ 2792.25.

- Frontend error logs (window):
  - Total error log rows: 178.
  - Earliest error: 2024-02-21 00:31:56+00:00; latest: 2024-02-21 00:32:13+00:00.
  - Example messages (earliest examples): "failed to retrieve ads" and many repeated "request error" entries clustered around 00:31:56–00:31:58.

Conclusion / interpretation:
- The frontend shows a sustained high socket count (26) and high request volume (row_count), plus many error logs concentrated around ~00:31:56–00:32:13. The immediate failure signal is the frontend producing many "request error" entries at that time. Investigate frontend request handling, socket/connection limits, and the downstream call that yields "failed to retrieve ads" (possible interaction with adservice) as the next steps.

The original code execution output of IPython Kernel is also provided below for reference:

(     kpi_name
0         cpu
1  latency-50
2  latency-90
3         mem
4      socket
5    workload,                    timestamp  value
0  2024-02-21 00:17:00+00:00   26.0
1  2024-02-21 00:18:00+00:00   26.0
2  2024-02-21 00:19:00+00:00   26.0
3  2024-02-21 00:20:00+00:00   26.0
4  2024-02-21 00:21:00+00:00   26.0
5  2024-02-21 00:22:00+00:00   26.0
6  2024-02-21 00:23:00+00:00   26.0
7  2024-02-21 00:24:00+00:00   26.0
8  2024-02-21 00:25:00+00:00   26.0
9  2024-02-21 00:26:00+00:00   26.0
10 2024-02-21 00:27:00+00:00   26.0
11 2024-02-21 00:28:00+00:00   26.0
12 2024-02-21 00:29:00+00:00   26.0
13 2024-02-21 00:30:00+00:00   26.0
14 2024-02-21 00:31:00+00:00   26.0
15 2024-02-21 00:32:00+00:00   26.0,           log_name  count             min_timestamp             max_timestamp  p95_value
0  log.error_count     16 2024-02-21 00:17:00+00:00 2024-02-21 00:32:00+00:00      64.00
1    log.row_count     16 2024-02-21 00:17:00+00:00 2024-02-21 00:32:00+00:00    2792.25, (178, Timestamp('2024-02-21 00:31:56+0000', tz='UTC'), Timestamp('2024-02-21 00:32:13+0000', tz='UTC')),                    timestamp                 message
0  2024-02-21 00:31:56+00:00  failed to retrieve ads
1  2024-02-21 00:31:56+00:00           request error
2  2024-02-21 00:31:56+00:00           request error
3  2024-02-21 00:31:56+00:00           request error
4  2024-02-21 00:31:56+00:00           request error
5  2024-02-21 00:31:56+00:00           request error
6  2024-02-21 00:31:57+00:00           request error
7  2024-02-21 00:31:57+00:00           request error
8  2024-02-21 00:31:57+00:00           request error
9  2024-02-21 00:31:57+00:00           request error
10 2024-02-21 00:31:57+00:00           request error
11 2024-02-21 00:31:57+00:00           request error
12 2024-02-21 00:31:57+00:00           request error
13 2024-02-21 00:31:57+00:00           request error
14 2024-02-21 00:31:57+00:00           request error
15 2024-02-21 00:31:57+00:00           request error
16 2024-02-21 00:31:57+00:00           request error
17 2024-02-21 00:31:57+00:00           request error
18 2024-02-21 00:31:58+00:00           request error
19 2024-02-21 00:31:58+00:00           request error)```
```